# Format sampling results

In [ ]:
import re
import sys
from ast import literal_eval

import pandas as pd
from rdkit import Chem

sys.path.append('..')

from modules.core.features.molecule_filter import MoleculeFilter

filter = MoleculeFilter()

In [ ]:
df = pd.read_csv('../data/sampling/filtering_comparison.csv')
df["smiles_after_filtering"] = df["smiles_after_filtering"].apply(literal_eval)
df.shape

In [ ]:
df.head()

In [ ]:
experts1 = pd.read_csv('../data/processed_all_custom_features/data_experts_1.csv')
experts2 = pd.read_csv('../data/raw/data_experts2.csv')
experts3 = pd.read_csv('../data/raw/data_experts3.csv')

# canonicalize smiles for all experts
experts1["smiles"] = experts1["smiles"].apply(lambda x: Chem.CanonSmiles(x))
experts2["smiles"] = experts2["smiles"].apply(lambda x: Chem.CanonSmiles(x))
experts3["smiles"] = experts3["smiles"].apply(lambda x: Chem.CanonSmiles(x))

In [ ]:
df_experts = pd.concat([experts1, experts2, experts3], ignore_index=True)["smiles"]
print(df_experts.shape)
df_experts = set(df_experts.to_list())
len(df_experts)

In [ ]:
def parse_filenames(df, column_name):
    """
    Parse a column of filenames into separate features.
    
    Args:
        df (pd.DataFrame): Input DataFrame.
        column_name (str): Name of the column to parse.
    
    Returns:
        pd.DataFrame: DataFrame with new feature columns.
    """
    # Define patterns to extract features
    def parse_row(row):
        pattern = {
            "model_name": r"(bionemo_[a-z]+|reinvent|filtered|symmetrical)",  # Model name
            "seed": r"(data_experts_\d+)",  # Seed information
            "num_samples": r"num_samples_(\d+)|(\d+smiles)",  # Number of samples (bionemo or reinvent style)
            "sampling_method": r"sampling_method_([\w-]+)",  # Sampling method
            "scaled_radius": r"scaled_radius_(\d+(\.\d+)?)",  # Scaled radius
            "chunks": r"(\d+x\d+)_chunks",  # Chunk info
            "beam_size": r"beam_size_(\d+)",  # Beam size
            "beam_alpha": r"beam_alpha_(\d+(\.\d+)?)",  # Beam alpha
            "top_k": r"top_k_(\d+)",  # Top-k value
            "top_p": r"top_p_(\d+(\.\d+)?)",  # Top-p value
            "temperature": r"temperature_(\d+(\.\d+)?)"  # Temperature
        }
        parsed_data = {}
        for key, regex in pattern.items():
            match = re.search(regex, row)
            if key == "model_name":
                # Remove "bionemo_" prefix if it's a bionemo model
                parsed_data[key] = match.group(1).replace("bionemo_", "") if match and "bionemo_" in match.group(1) else match.group(1)
                # if name is filtered, then it is a reinvent
                if parsed_data[key] == "filtered" or parsed_data[key] == "symmetrical":
                    parsed_data[key] = "reinvent"
            elif key == "seed":
                # Extract seed (data_experts_x)
                parsed_data[key] = match.group(1) if match else None
            elif key == "num_samples":
                if match:  # Check if a match is found
                    num_match = match.group(1) or match.group(2)
                    parsed_data[key] = re.sub(r"smiles", "", num_match) if num_match else None
                else:
                    parsed_data[key] = None if not match else match.group(1)
            else:
                parsed_data[key] = match.group(1) if match else None
        return parsed_data

    # Apply parsing logic to each row
    parsed_data = df[column_name].apply(parse_row).apply(pd.Series)
    
    # Concatenate original DataFrame with parsed columns
    df = pd.concat([df, parsed_data], axis=1)
    return df


df = parse_filenames(df, "filenames")
# remove duplicate columns
df = df.loc[:, ~df.columns.duplicated()]
df.to_csv('../data/sampling/filtering_comparison_parsed.csv', index=False)
# # group by model and mean over the number of samples
df[["model_name", "num_total_molecules", "num_filtered_molecules"]].groupby("model_name").mean()

In [ ]:
# group by model name and aggregate molecules after filtering
res_df = df.groupby("model_name")["smiles_after_filtering"].agg(lambda x: [item for sublist in x for item in sublist])
# apply set to remove duplicates
res_df = res_df.apply(set)


print(res_df.apply(len))

# exclude experts from the set
res_df = res_df.apply(lambda x: x - df_experts)

print(res_df.apply(len))

In [ ]:
res_df.to_csv('../data/sampling/xlsx_input.csv', index=True)

In [ ]:
# TODO: execute script to generate excel

## Experts data which passed the filters
Go to root as MoleculeFilter has paths relative to root 

In [ ]:
cd .. 

In [ ]:
experts_dict = {
    "experts1": experts1.copy(),
    "experts2": experts2.copy(),
    "experts3": experts3.copy()
}

for k, v in experts_dict.items():
    v['passed'] = False  # Initialize 'passed' column for the current DataFrame

    smiles_list = v['smiles'].to_list()  # Extract SMILES as a list (only once)

    passed_smiles = set(filter.apply(smiles_list)) # convert passed to set for O(1) lookup
    print(f"Number of passed SMILES: {len(passed_smiles)}, Passed SMILES: {passed_smiles}")

    v['passed'] = [sml in passed_smiles for sml in smiles_list]  # Efficiently set 'passed' using a list comprehension

In [ ]:
import os
import shutil

import pandas as pd
from openpyxl import Workbook
from openpyxl.drawing.image import Image
from rdkit import Chem
from rdkit.Chem import Draw


def generate_excel_from_dict(
    model_data: dict, output_file: str, output_folder: str = "temp_images"
):
    """
    Generates an Excel file with SMILES structures, images, and an optional
    column indicating if the SMILES passed the filter, directly from a
    dictionary input.

    Args:
        model_data (dict): Dictionary of models and their SMILES DataFrames,
            where DataFrames can optionally contain a 'passed' column.
        output_file (str): Path to save the output Excel file.
        output_folder (str): Path to the folder for temporary images.
            Defaults to 'temp_images'.

    Returns:
        None
    """
    # Create a new Excel workbook
    wb = Workbook()

    # Create the output folder for temporary images
    os.makedirs(output_folder, exist_ok=True)

    for model_name, df in model_data.items():
        # Create a new sheet for each model
        ws = wb.create_sheet(title=model_name)

        # Add headers
        ws.cell(row=1, column=1, value="SMILES")
        ws.cell(row=1, column=2, value="Structure")
        ws.cell(row=1, column=3, value="Grade (0-5)")

        has_passed = "passed" in df.columns
        if has_passed:
            ws.cell(row=1, column=4, value="Passed Filter")

        # Set default column widths
        ws.column_dimensions["A"].width = 30  # SMILES column
        ws.column_dimensions["B"].width = 50  # Image column
        ws.column_dimensions["C"].width = 20  # Grade column
        if has_passed:
            ws.column_dimensions["D"].width = 15  # Passed column

        # Set default row height
        for row in range(2, len(df) + 2):  # Adjust for data rows
            ws.row_dimensions[row].height = 250

        # Process each SMILES in the list
        for i, row in enumerate(df.iterrows(), start=2):
            _, data = row
            # Parse the molecule
            smiles = data["smiles"]
            mol = Chem.MolFromSmiles(smiles)

            # Add SMILES to the Excel
            ws.cell(row=i, column=1, value=smiles)

            img_path = os.path.join(output_folder, f"{model_name}_mol_{i}.png")
            Draw.MolToFile(mol, img_path)

            # Add the image to the Excel
            img = Image(img_path)
            ws.add_image(img, f"B{i}")

            # Add grading instructions
            ws.cell(row=i, column=3, value="Enter 0-5")
            if has_passed:
                ws.cell(row=i, column=4, value=data["passed"])

    # Remove the default sheet
    if "Sheet" in wb.sheetnames:
        del wb["Sheet"]

    # Save the Excel file
    wb.save(output_file)
    print(f"Excel file saved as {output_file}.")
    if os.path.exists(output_folder):
         shutil.rmtree(output_folder)
         print(f"Temporary folder '{output_folder}' has been deleted.")



generate_excel_from_dict(experts_dict, "experts_filters.xlsx")